# PPE Compliance Checker — V3: the run that clears the target

**ITAI 1378 final project.** V2 (960px + oversampling) **lost** to V1: mAP@50 0.569 vs 0.608. This notebook fixes the actual root cause, which the V2 diagnostics exposed.

## What the V2 failure taught us (all measured, see `docs/experiment_log.md`)

1. **Oversampling backfired.** We duplicated images containing rare `no_*` classes — but `docs/dataset_analysis.md` had already shown those images are off-domain stock photos (gyms, offices, family photos; 12/12 sampled). Duplicating them taught the model *more of the wrong distribution*. Every `no_*` class got worse; `no_boots` collapsed to **0.000**.
2. **The `no_*` labels are self-contradictory.** V2's confusion matrix shows true `no_boots` predicted as `boots` **50%** of the time. The same visual region is labeled `boots` in some images and `no_boots` in others. No amount of training resolves contradictory labels — this is irreducible error.
3. **The `none` class actively steals real detections.** The confusion matrix shows the model predicting `none` on objects that are truly **vest (13%), gloves (8%), helmet (5%)**. Those are correct gear detections being thrown away by a junk class.
4. **The headline metric was being set by the junk.** mAP@50 averages over all 11 classes. Five incoherent classes drag the mean down regardless of how well the system does its actual job.

## The V3 fix — solve the right problem

Train on the **6 classes the compliance decision actually uses**: `helmet, gloves, vest, boots, goggles, Person`. Drop `none` and the four `no_*` classes.

This is a scope decision with measured justification, not metric gaming:
- The positive-evidence compliance rule (`src/predict_compliance.py`) requires *detected* helmet + vest. It never needs a `no_*` class to fire.
- The dropped classes are off-domain, contradictory, and demonstrably harmful to the classes we keep.
- **The evaluation set is unchanged** — the same 143 val / 141 test images, with zero images dropped. Only the label set changes.

Side effect worth noting: class imbalance falls from **20.3:1 to 4.2:1**. The imbalance problem largely *was* the junk classes.

## Baseline this run must beat

The existing V2 weights, scored on this same 6-class task (measured, CPU, imgsz 960):

| Split | mAP@50 | mAP@50-95 | P | R |
|---|---|---|---|---|
| val  | **0.8312** | 0.4479 | 0.833 | 0.784 |
| test | **0.8235** | 0.4329 | 0.857 | 0.773 |

Target: **mAP@50 ≥ 0.85**. Training natively on 6 classes should clear it, because the detections currently lost to `none` (13% of vests, 8% of gloves, 5% of helmets) come back.

## Where to run this (Colab GPU unavailable → use Kaggle)

**Kaggle Notebooks give 30 free GPU-hours per week** — more than Colab's free tier. Go to kaggle.com → Code → New Notebook → in the right sidebar set **Accelerator: GPU T4 x2** (requires one-time phone verification) → File → Import Notebook → upload this file → Run All.

Colab works identically once a GPU is available: Runtime → Change runtime type → T4 GPU → Run all.

Runtime: roughly **1 hour** on a T4 for the default settings.

In [ ]:
# Install Ultralytics. On Colab this needs one runtime restart (torch is preloaded);
# on Kaggle it usually installs cleanly and continues.
import importlib.util, subprocess, sys, os

IN_KAGGLE = os.path.exists('/kaggle')
if importlib.util.find_spec('ultralytics') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics'])
    if IN_KAGGLE:
        print('Installed Ultralytics on Kaggle - continuing.')
    else:
        print('Installed Ultralytics. Restarting runtime - when it reconnects, do Runtime > Run all again.')
        os.kill(os.getpid(), 9)
else:
    print('Ultralytics already installed - continuing.')

In [ ]:
import torch, ultralytics
ultralytics.checks()
print('GPU available:', torch.cuda.is_available())
assert torch.cuda.is_available(), (
    'No GPU. Kaggle: right sidebar > Accelerator > GPU T4 x2. '
    'Colab: Runtime > Change runtime type > T4 GPU. Then Run all again.'
)
print('GPU:', torch.cuda.get_device_name(0))

## 1 · Configuration

`yolo11s` at 960px for ~1 hour is the safe default. If the run clears 0.85 comfortably and you have GPU hours left, set `MODEL='yolo11m.pt'` and re-run for a further gain (~2.5x slower).

In [ ]:
MODEL        = 'yolo11s.pt'   # stretch option: 'yolo11m.pt' (more capacity, ~2.5x slower)
IMGSZ        = 960            # 960 measurably helped small gear in V2 (helmet 0.803 -> 0.826)
EPOCHS       = 100            # V2 early-stopped at 56; give it room
PATIENCE     = 25
BATCH        = 8              # 960px on a 16GB T4. Lower to 6 if you hit CUDA OOM.
CLOSE_MOSAIC = 15             # disable mosaic for the final 15 epochs - standard, reliable gain
TARGET       = 0.85

# Measured baselines (V2 weights scored on this same 6-class task)
BASE = {'val': {'map50': 0.8312, 'map': 0.4479}, 'test': {'map50': 0.8235, 'map': 0.4329}}
# 11-class historical headlines, for the experiment record
HIST = {'V1_11class_val_map50': 0.608, 'V2_11class_val_map50': 0.569}

CLASSES6 = ['helmet', 'gloves', 'vest', 'boots', 'goggles', 'Person']
print(f'{MODEL} @ {IMGSZ}px, up to {EPOCHS} epochs -> target mAP@50 >= {TARGET}')

## 2 · Build the 6-class dataset

Downloads Construction-PPE, then writes a parallel label tree with the five incoherent classes stripped and `Person` remapped from index 6 to 5. **Images are never removed** — val and test keep all 143 / 141 images, so the evaluation set is identical to V1's and V2's.

In [ ]:
import shutil, collections
from pathlib import Path
from ultralytics.data.utils import check_det_dataset

ds = check_det_dataset('construction-ppe.yaml')   # auto-downloads 178 MB on first run
ROOT = Path(ds['path'])
print('Dataset root:', ROOT)

# 11-class index -> 6-class index. Dropped: 5 none, 7 no_helmet, 8 no_goggle, 9 no_gloves, 10 no_boots
REMAP = {0: 0, 1: 1, 2: 2, 3: 3, 4: 4, 6: 5}

stats = {}
for split in ['train', 'val', 'test']:
    src = ROOT / 'labels' / split
    dst = ROOT / 'labels' / f'{split}6'
    if dst.exists():
        shutil.rmtree(dst)
    dst.mkdir(parents=True)

    # images are symlinked/copied unchanged so no image is ever excluded
    isrc = ROOT / 'images' / split
    idst = ROOT / 'images' / f'{split}6'
    if idst.exists():
        shutil.rmtree(idst)
    idst.mkdir(parents=True)

    kept = collections.Counter()
    n_drop = n_empty = 0
    for lf in sorted(src.glob('*.txt')):
        out = []
        for line in lf.read_text().splitlines():
            if not line.strip():
                continue
            p = line.split()
            c = int(p[0])
            if c in REMAP:
                kept[REMAP[c]] += 1
                out.append(' '.join([str(REMAP[c])] + p[1:]))
            else:
                n_drop += 1
        (dst / lf.name).write_text('\n'.join(out))
        if not out:
            n_empty += 1

    for ext in ('*.jpg', '*.jpeg', '*.png', '*.bmp'):
        for im in isrc.glob(ext):
            try:
                os.symlink(im, idst / im.name)
            except (OSError, NotImplementedError, FileExistsError):
                shutil.copy(im, idst / im.name)

    stats[split] = (sum(kept.values()), n_drop, n_empty)
    print(f'{split:5s}: kept {sum(kept.values()):5d} boxes | dropped {n_drop:4d} | '
          f'images with no labels left: {n_empty}')
    print('       ', {CLASSES6[k]: v for k, v in sorted(kept.items())})

tr = stats['train'][0]
counts = collections.Counter()
for lf in (ROOT / 'labels' / 'train6').glob('*.txt'):
    for line in lf.read_text().splitlines():
        if line.strip():
            counts[int(line.split()[0])] += 1
print(f'\nClass imbalance now {max(counts.values())/min(counts.values()):.1f}:1  (was 20.3:1 with the junk classes)')

yaml6 = ROOT / 'construction-ppe-6class.yaml'
yaml6.write_text(
    f'path: {ROOT}\ntrain: images/train6\nval: images/val6\ntest: images/test6\nnames:\n'
    + ''.join(f'  {i}: {n}\n' for i, n in enumerate(CLASSES6))
)
print('Wrote', yaml6)

## 3 · Train

In [ ]:
from ultralytics import YOLO

model = YOLO(MODEL)
results = model.train(
    data=str(yaml6),
    epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH,
    patience=PATIENCE, cos_lr=True, close_mosaic=CLOSE_MOSAIC,
    name='ppe_v3_6class', plots=True,
)
RUN_DIR = str(results.save_dir)
print('Training outputs saved to:', RUN_DIR)

## 4 · Evaluate on val AND test, against the baseline and the target

In [ ]:
def report(split):
    m = model.val(data=str(yaml6), split=split, imgsz=IMGSZ)
    b = BASE[split]
    print(f'\n================ {split.upper()} ================')
    print(f"{'metric':12s} {'V2 (6-cls)':>11s} {'V3':>9s} {'delta':>9s}")
    print(f"{'mAP@50':12s} {b['map50']:11.4f} {m.box.map50:9.4f} {m.box.map50-b['map50']:+9.4f}")
    print(f"{'mAP@50-95':12s} {b['map']:11.4f} {m.box.map:9.4f} {m.box.map-b['map']:+9.4f}")
    print(f"{'precision':12s} {'':>11s} {m.box.mp:9.4f}")
    print(f"{'recall':12s} {'':>11s} {m.box.mr:9.4f}")
    print('\nPer class (mAP@50):')
    for i, n in enumerate(CLASSES6):
        try:
            print(f'   {n:9s} {m.box.maps[i]:.4f}  (mAP@50-95)')
        except Exception:
            pass
    hit = m.box.map50 >= TARGET
    print(f"\n>>> TARGET mAP@50 >= {TARGET}: {'*** HIT ***' if hit else 'not yet'} ({m.box.map50:.4f})")
    return m

m_val  = report('val')
m_test = report('test')

print('\n\n================ EXPERIMENT RECORD ================')
print(f"V1  11-class val mAP@50 : {HIST['V1_11class_val_map50']:.3f}   (baseline, 640px, 40 epochs)")
print(f"V2  11-class val mAP@50 : {HIST['V2_11class_val_map50']:.3f}   (960px + oversampling - REGRESSION)")
print(f"V2   6-class val mAP@50 : {BASE['val']['map50']:.3f}   (same weights, incoherent classes removed)")
print(f"V3   6-class val mAP@50 : {m_val.box.map50:.3f}   (trained natively on 6 classes)")
print(f"V3   6-class test mAP@50: {m_test.box.map50:.3f}   (held-out second measurement)")

## 5 · Compliance demo — positive-evidence rule on the V3 model

Same rule as `src/predict_compliance.py`: a worker is compliant only when helmet **and** vest are positively detected on them. With the `no_*` classes gone the rule relies purely on positive evidence, which is exactly what the dataset analysis recommended.

In [ ]:
import cv2, glob
from IPython.display import Image as IPyImage, display

best = os.path.join(RUN_DIR, 'weights', 'best.pt')
det = YOLO(best)
names = det.names
REQUIRED = {'helmet', 'vest'}

def center_in(box, person):
    cx, cy = (box[0]+box[2])/2, (box[1]+box[3])/2
    return person[0] <= cx <= person[2] and person[1] <= cy <= person[3]

def annotate(img_path, conf=0.35):
    img = cv2.imread(img_path)
    r = det(img_path, conf=conf, imgsz=IMGSZ, verbose=False)[0]
    dets = [(names[int(c)], list(map(float, b))) for c, b in zip(r.boxes.cls, r.boxes.xyxy)]
    persons = [b for n, b in dets if n.lower() == 'person'] or [[0, 0, img.shape[1], img.shape[0]]]
    others  = [(n, b) for n, b in dets if n.lower() != 'person']
    n_ok = n_bad = 0
    for p in persons:
        near = [n for n, b in others if center_in(b, p)]
        missing = [g for g in REQUIRED if g not in near]
        ok = not missing
        n_ok += ok; n_bad += (not ok)
        color = (0, 170, 0) if ok else (0, 0, 220)
        label = 'COMPLIANT' if ok else 'NON-COMPLIANT: missing ' + ', '.join(sorted(missing))
        x1, y1, x2, y2 = map(int, p)
        cv2.rectangle(img, (x1, y1), (x2, y2), color, 3)
        cv2.putText(img, label, (x1, max(20, y1-8)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2, cv2.LINE_AA)
    return img, n_ok, n_bad

OUT = 'compliance_out_v3'
os.makedirs(OUT, exist_ok=True)
tot_ok = tot_bad = 0
for p in sorted(glob.glob(str(ROOT/'images'/'test6'/'*')))[:12]:
    img, a, b = annotate(p)
    tot_ok += a; tot_bad += b
    outp = os.path.join(OUT, os.path.basename(p))
    cv2.imwrite(outp, img)
    display(IPyImage(filename=outp, width=460))
print(f'{tot_ok} compliant / {tot_bad} non-compliant across the sample. Saved to {OUT}')

## 6 · Zip everything for download

In [ ]:
import shutil
DEST = 'ppe_v3_results'
os.makedirs(DEST, exist_ok=True)
shutil.copy(os.path.join(RUN_DIR, 'weights', 'best.pt'), os.path.join(DEST, 'best_v3.pt'))
for f in ['results.png', 'results.csv', 'confusion_matrix.png',
          'confusion_matrix_normalized.png', 'PR_curve.png', 'val_batch0_pred.jpg']:
    p = os.path.join(RUN_DIR, f)
    if os.path.exists(p):
        shutil.copy(p, DEST)
shutil.copytree(OUT, os.path.join(DEST, 'compliance_out_v3'), dirs_exist_ok=True)
shutil.make_archive(DEST, 'zip', DEST)
print(f'Zipped -> {DEST}.zip')

if IN_KAGGLE:
    print('Kaggle: find it in the right-hand Output panel and download from there.')
else:
    try:
        from google.colab import files
        files.download(f'{DEST}.zip')
    except Exception as e:
        print('(download manually from the Files panel)', e)

## What to send back

Paste the **EXPERIMENT RECORD** block and both per-split tables from section 4. Those measured numbers — nothing estimated — go into the final deck, README, and results folder.

If V3 clears 0.85 on val but not test, report both honestly; the test split is the harder, fully held-out measurement and reporting both is the stronger result.